# Recap of Spark Architecture:

SparkContext runs on the Driver Program (which runs on the master node), communicates with Cluster Manager to ask for resources (mem, core). \

SparkContext directly communicates with Executors (run at Slave nodes), sending jobs/data, receiving data. 

![It's the Jews](https://spark.apache.org/docs/latest/img/cluster-overview.png)

In [1]:
from pyspark import SparkContext, SparkConf
# Initialize SparkContext with a specific application name and master URL, and define driver's max core count, memory, and executor memory, cores, and instances.
conf = (SparkConf().setAppName("MyApp").setMaster("spark://10.1.11.5:7077")  # For local, replace with "local[x]" (x = # of cores, or * if use all)
    .set("spark.driver.memory", "4g")   # Max memory for driver, works for all CM
    .set("spark.driver.cores", "2") # Max # of cores for driver, works if CM = yarn/k8s. Not work if CM = standalone/mesos/local (use all cores by default)
    .set("spark.executor.memory", "4g")  # Max memory for each executor, works for all CM
    .set("spark.executor.cores", "4") # Max # of cores for each executor, works for all CM
    .set("spark.executor.instances", "2") # Max # of executors, works if CM = standalone/mesos. Not work if CM = yarn/k8s (use dynamic allocation by default)
    .set("spark.cores.max", "16")) # Max # of cores (global setting), must be divisible by spark.executor.cores, only works on standalone/mesos
sc = SparkContext(conf=conf).getOrCreate()


:: loading settings :: url = jar:file:/home/hpcc/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/hpcc/.ivy2.5.2/cache
The jars for the packages stored in: /home/hpcc/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
org.apache.spark#spark-streaming-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-19e20765-8a8a-462f-a0f4-3b1f6b11ef77;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found com.github.luben#zstd-jni;1.5.6-9 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	fou

## New way of initializing Spark

In [11]:
from pyspark.sql import SparkSession # = SparkContext + Spark's SQL + Spark Streaming +...
spark = (SparkSession.builder
    .appName("MyApp")  # Application name
    .master("local[*]")  # Master URL
    .config("spark.driver.memory", "4g")   # same as above
    .getOrCreate()
)



26/10/29 16:50:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [12]:
print(spark.sparkContext.uiWebUrl)

http://10.229.189.227:4041


# Create a simple DataFrame

In [4]:
# Generate simple dataframe of customer buying groceries data for reccommendation system
data = [("Abraham", "Apples", 5),
        ("Jacob", "Bananas", 3),
        ("Jacob", "Oranges", 2),
        ("Benjamin", "Apples", 2),
        ("David", "Bananas", 2),
        ("David", "Oranges", 5),
        ("Esau", "Apples", 3),
        ("Esau", "Bananas", 5),         
        ("Ishmael", "Oranges", 4),
        ("Samael", "Apples", 4),
        ("Samael", "Bananas", 1),       
        ("Samael", "Oranges", 1)]
columns = ["Customer", "Item", "Quantity"]
df = spark.createDataFrame(data, columns)

In [5]:
df.show()  # Show the dataframe, similar to pandas df.head()
print(df.count())
df.printSchema()  # Print the schema of the dataframe, similar to pandas df.info()

+--------+-------+--------+
|Customer|   Item|Quantity|
+--------+-------+--------+
| Abraham| Apples|       5|
|   Jacob|Bananas|       3|
|   Jacob|Oranges|       2|
|Benjamin| Apples|       2|
|   David|Bananas|       2|
|   David|Oranges|       5|
|    Esau| Apples|       3|
|    Esau|Bananas|       5|
| Ishmael|Oranges|       4|
|  Samael| Apples|       4|
|  Samael|Bananas|       1|
|  Samael|Oranges|       1|
+--------+-------+--------+

12
root
 |-- Customer: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: long (nullable = true)



In [6]:
# Total Apples sold? 
from pyspark.sql.functions import col, sum
"""
Approach, filter apple purchases -> aggregate by quantity, collect (the distributed result) to driver node
"""
total_apples = df.filter(col("Item") == "Apples").agg(sum("Quantity").alias("Total_Apples")).collect()[0]["Total_Apples"]
print(f"Total Apples sold: {total_apples}")
"""
Approach 2: group by item -> aggregate by quantity -> filter apple purchases, collect (the distributed result) to driver node
"""
total_apples_2 = df.groupBy("Item").agg(sum("Quantity").alias("Total_Quantity")).filter(col("Item") == "Apples").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 2): {total_apples_2}")
"""Approach 3 and 4: same as first and second approach, but use SQL syntax"""
df.createOrReplaceTempView("purchases")  # Create a temporary view for SQL queries
total_apples_3 = spark.sql("SELECT SUM(Quantity) AS Total_Apples FROM purchases WHERE Item = 'Apples'").collect()[0]["Total_Apples"]
print(f"Total Apples sold (approach 3): {total_apples_3}")
total_apples_4 = spark.sql("SELECT Item, SUM(Quantity) AS Total_Quantity FROM purchases GROUP BY Item HAVING Item = 'Apples'").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 4): {total_apples_4}")

Total Apples sold: 14
Total Apples sold (approach 2): 14
Total Apples sold (approach 3): 14
Total Apples sold (approach 4): 14


In [7]:
# Create another price table to join with purchases
price_data = [("Apples", 1.0),
              ("Bananas", 0.5),
              ("Oranges", 0.8)]
price_columns = ["Item", "Price"]
price_df = spark.createDataFrame(price_data, price_columns)
price_df.show()

+-------+-----+
|   Item|Price|
+-------+-----+
| Apples|  1.0|
|Bananas|  0.5|
|Oranges|  0.8|
+-------+-----+



In [8]:
# How much did Jacob spend on groceries?
from pyspark.sql.functions import expr
# Join purchases with prices on Item column
joined_df = df.join(price_df, on="Item", how="inner")
# Calculate total cost for each purchase
joined_df = joined_df.withColumn("Total_Cost", expr("Quantity * Price"))
# Filter for Jacob's purchases and sum the total cost
jacob_total = joined_df.filter(col("Customer") == "Jacob").agg(sum("Total_Cost").alias("Jacob_Total")).collect()[0]["Jacob_Total"]
print(f"Jacob's total grocery spending: ${jacob_total:.2f}")

Jacob's total grocery spending: $3.10


# Save DataFrame to MinIO

Distributed Storage System

In [9]:
# Add Spark Context on the go, add MinIO, and change committer to "magic" for better performance
# WARNING, if config is set directly in hadoop, do not add spark.hadoop prefix, only add them during SparkConf/SparkSession initialization

spark.sparkContext._jsc.hadoopConfiguration().set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.4.1") # Library for connecting to AWS S3/MinIO
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://10.1.11.5:9000") # MinIO endpoint (location)
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.access.key", "test") # username
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "hcmuthpcc") # password
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true") # Enable path style access (http://10.1.11.5:9000/big-data/test/... instead of http://big-data.endpoint/test/...)
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # Use basic credentials that we set above
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Use S3A file system
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.name", "magic")                                            # Use magic committer, better than default FileOutputCommitter
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.magic.partitioned.enabled", "true")                        # Enable partitioned writing
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.fast.upload", "true")                                                # Enable fast upload
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.fast.upload.buffer", "disk")                                         # Use disk buffer for fast upload   # OPTIONAL
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.staging.conflict-mode", "replace")                         # Overwrite existing files          # OPTIONAL
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.staging.abort.pending.uploads", "true")                    # Abort pending uploads             # OPTIONAL

# Write to MinIO
df.write.mode("overwrite").csv("s3a://big-data/test/jacob.csv", header=True)

Py4JJavaError: An error occurred while calling o149.csv.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2724)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:135)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWritingFileFormat(DataSource.scala:485)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWriting(DataSource.scala:589)
	at org.apache.spark.sql.classic.DataFr

In [9]:
# Check if the file is written by reading it back
minio_df = spark.read.csv("s3a://big-data/test/jacob.csv", header=True, inferSchema=True)
minio_df.show()

+--------+-------+--------+
|Customer|   Item|Quantity|
+--------+-------+--------+
|Benjamin| Apples|       2|
| Ishmael|Oranges|       4|
| Abraham| Apples|       5|
|  Samael|Bananas|       1|
|  Samael|Oranges|       1|
|   Jacob|Bananas|       3|
|   Jacob|Oranges|       2|
|   David|Bananas|       2|
|   David|Oranges|       5|
|  Samael| Apples|       4|
|    Esau|Bananas|       5|
|    Esau| Apples|       3|
+--------+-------+--------+



In [10]:
spark.stop()